# AMR-UnderDifferentNoises-DL: Sonuç Analizi ve Karşılaştırma

Bu notebook, fine-tuning işlemi sonrasında elde edilen `.pkl` uzantılı sonuç dosyalarını (Accuracy, F1 Score, MCC) Google Drive'dan okur ve **modeller arası ile veri setleri arası kıyaslamaları** tek bir grafik üzerinde birleştirir.
Ayrıca oluşturulan bu birleşik grafikleri yüksek çözünürlüklü olarak Drive'ınıza kaydeder.

In [ ]:
# 1. Drive'a Bağlan ve Gerekli Kütüphaneleri Yükle
from google.colab import drive
drive.mount('/content/drive')

import os
import pickle
import numpy as np
import matplotlib.pyplot as plt

RESULTS_DIR = '/content/drive/MyDrive/AMR-Project/fine_tuning_results'
if not os.path.exists(RESULTS_DIR):
    RESULTS_DIR = '/content/drive/MyDrive/AMR-Project/results'

SAVE_DIR = os.path.join(RESULTS_DIR, 'Karsilastirma_Grafikleri')
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"\nSonuçların okunacağı dizin: {RESULTS_DIR}")
print(f"Grafiklerin otomatik kaydedileceği dizin: {SAVE_DIR}")


In [ ]:
# 2. Verileri Okuma Fonksiyonu
def load_pkl_data(model, dataset, metric_file='acc.pkl'):
    folder_name = f"{model}_{dataset}"
    file_path = os.path.join(RESULTS_DIR, folder_name, 'predictions', metric_file)
    if not os.path.exists(file_path):
        file_path = os.path.join(RESULTS_DIR, folder_name, metric_file)
    if os.path.exists(file_path):
        with open(file_path, 'rb') as f:
            return pickle.load(f)
    else:
        return None

models = ['mcldnn', 'petcgdnn']
datasets = ['rayleigh', 'rician_k3', 'rician_k10']


## A) CLASSIFICATION ACCURACY (Doğruluk Oranı) Analizleri

In [ ]:
# 3. Model Karşılaştırması (ACCURACY)
def plot_model_comparison_acc(dataset_name, dataset_title):
    mcldnn_acc = load_pkl_data('mcldnn', dataset_name, 'acc.pkl')
    petcgdnn_acc = load_pkl_data('petcgdnn', dataset_name, 'acc.pkl')
    if mcldnn_acc is None or petcgdnn_acc is None: return
        
    snrs_sorted = sorted(list(mcldnn_acc.keys()))
    mcl_vals = [mcldnn_acc[s]*100 for s in snrs_sorted]
    pet_vals = [petcgdnn_acc[s]*100 for s in snrs_sorted]
    
    plt.figure(figsize=(10, 6), dpi=150)
    plt.plot(snrs_sorted, mcl_vals, 'o-', linewidth=2.5, label='MCLDNN', color='#E63946')
    plt.plot(snrs_sorted, pet_vals, 's-', linewidth=2.5, label='PET-CGDNN', color='#1D3557')
    plt.xlabel("Signal to Noise Ratio (SNR) in dB", fontsize=12)
    plt.ylabel("Classification Accuracy (%)", fontsize=12)
    plt.title(f"Model Accuracy Comparison on {dataset_title} Channel", fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([0, 105])
    plt.savefig(os.path.join(SAVE_DIR, f"model_comp_accuracy_{dataset_name}.png"), bbox_inches='tight')
    plt.show()

print("========== MODEL KARŞILAŞTIRMALARI (ACCURACY) ==========")
plot_model_comparison_acc('rayleigh', 'Rayleigh Fading')
plot_model_comparison_acc('rician_k3', 'Rician Fading (K=3)')
plot_model_comparison_acc('rician_k10', 'Rician Fading (K=10)')


In [ ]:
# 4. Kanal Karşılaştırması (ACCURACY)
def plot_channel_comparison_acc(model_name, model_title):
    rayleigh_acc = load_pkl_data(model_name, 'rayleigh', 'acc.pkl')
    rician3_acc = load_pkl_data(model_name, 'rician_k3', 'acc.pkl')
    rician10_acc = load_pkl_data(model_name, 'rician_k10', 'acc.pkl')
    if rayleigh_acc is None or rician3_acc is None or rician10_acc is None: return
        
    snrs_sorted = sorted(list(rayleigh_acc.keys()))
    plt.figure(figsize=(10, 6), dpi=150)
    plt.plot(snrs_sorted, [rayleigh_acc[s]*100 for s in snrs_sorted], 'o-', linewidth=2.5, label='Rayleigh', color='#E63946')
    plt.plot(snrs_sorted, [rician3_acc[s]*100 for s in snrs_sorted], 's-', linewidth=2.5, label='Rician K=3', color='#2A9D8F')
    plt.plot(snrs_sorted, [rician10_acc[s]*100 for s in snrs_sorted], '^-', linewidth=2.5, label='Rician K=10', color='#457B9D')
    plt.xlabel("SNR (dB)", fontsize=12)
    plt.ylabel("Classification Accuracy (%)", fontsize=12)
    plt.title(f"Channel Fading Impact on {model_title} (Accuracy)", fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([0, 105])
    plt.savefig(os.path.join(SAVE_DIR, f"channel_comp_accuracy_{model_name}.png"), bbox_inches='tight')
    plt.show()

print("========== KANAL KARŞILAŞTIRMALARI (ACCURACY) ==========")
plot_channel_comparison_acc('mcldnn', 'MCLDNN')
plot_channel_comparison_acc('petcgdnn', 'PET-CGDNN')


## B) F1 SCORE (Macro) Analizleri

In [ ]:
# 5. Model Karşılaştırması (F1 SCORE)
def plot_model_comparison_f1(dataset_name, dataset_title):
    mcl_f1 = load_pkl_data('mcldnn', dataset_name, 'f1_macro_scores.pkl')
    pet_f1 = load_pkl_data('petcgdnn', dataset_name, 'f1_macro_scores.pkl')
    if mcl_f1 is None or pet_f1 is None: return
        
    snrs_sorted = sorted(list(mcl_f1.keys()))
    plt.figure(figsize=(10, 6), dpi=150)
    plt.plot(snrs_sorted, [mcl_f1[s] for s in snrs_sorted], 'o-', linewidth=2.5, label='MCLDNN', color='#E63946')
    plt.plot(snrs_sorted, [pet_f1[s] for s in snrs_sorted], 's-', linewidth=2.5, label='PET-CGDNN', color='#1D3557')
    plt.xlabel("SNR (dB)", fontsize=12)
    plt.ylabel("F1 Score (Macro)", fontsize=12)
    plt.title(f"Model F1 Score Comparison on {dataset_title} Channel", fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([0, 1.05])
    plt.savefig(os.path.join(SAVE_DIR, f"model_comp_f1_{dataset_name}.png"), bbox_inches='tight')
    plt.show()

print("========== MODEL KARŞILAŞTIRMALARI (F1 SCORE) ==========")
plot_model_comparison_f1('rayleigh', 'Rayleigh Fading')
plot_model_comparison_f1('rician_k3', 'Rician Fading (K=3)')
plot_model_comparison_f1('rician_k10', 'Rician Fading (K=10)')


In [ ]:
# 5b. Kanal Karşılaştırması (F1 SCORE)
def plot_channel_comparison_f1(model_name, model_title):
    rayleigh_f1  = load_pkl_data(model_name, 'rayleigh',   'f1_macro_scores.pkl')
    rician3_f1   = load_pkl_data(model_name, 'rician_k3',  'f1_macro_scores.pkl')
    rician10_f1  = load_pkl_data(model_name, 'rician_k10', 'f1_macro_scores.pkl')
    if rayleigh_f1 is None or rician3_f1 is None or rician10_f1 is None: return

    snrs_sorted = sorted(list(rayleigh_f1.keys()))
    plt.figure(figsize=(10, 6), dpi=150)
    plt.plot(snrs_sorted, [rayleigh_f1[s]  for s in snrs_sorted], 'o-', linewidth=2.5, label='Rayleigh',    color='#E63946')
    plt.plot(snrs_sorted, [rician3_f1[s]   for s in snrs_sorted], 's-', linewidth=2.5, label='Rician K=3',  color='#2A9D8F')
    plt.plot(snrs_sorted, [rician10_f1[s]  for s in snrs_sorted], '^-', linewidth=2.5, label='Rician K=10', color='#457B9D')
    plt.xlabel('SNR (dB)', fontsize=12)
    plt.ylabel('F1 Score (Macro)', fontsize=12)
    plt.title(f'Channel Fading Impact on {model_title} (F1 Score)', fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([0, 1.05])
    plt.savefig(os.path.join(SAVE_DIR, f'channel_comp_f1_{model_name}.png'), bbox_inches='tight')
    plt.show()

print('========== KANAL KARSILASTIRMALARI (F1 SCORE) ==========')
plot_channel_comparison_f1('mcldnn',   'MCLDNN')
plot_channel_comparison_f1('petcgdnn', 'PET-CGDNN')


## C) MCC (Matthews Correlation Coefficient) Analizleri

In [ ]:
# 6. Model Karşılaştırması (MCC)
def plot_model_comparison_mcc(dataset_name, dataset_title):
    mcl_mcc = load_pkl_data('mcldnn', dataset_name, 'mcc_metric_scores.pkl')
    pet_mcc = load_pkl_data('petcgdnn', dataset_name, 'mcc_metric_scores.pkl')
    if mcl_mcc is None or pet_mcc is None: return
        
    snrs_sorted = sorted(list(mcl_mcc.keys()))
    plt.figure(figsize=(10, 6), dpi=150)
    plt.plot(snrs_sorted, [mcl_mcc[s] for s in snrs_sorted], 'o-', linewidth=2.5, label='MCLDNN', color='#E63946')
    plt.plot(snrs_sorted, [pet_mcc[s] for s in snrs_sorted], 's-', linewidth=2.5, label='PET-CGDNN', color='#1D3557')
    plt.xlabel("SNR (dB)", fontsize=12)
    plt.ylabel("MCC", fontsize=12)
    plt.title(f"Model MCC Comparison on {dataset_title} Channel", fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([-0.1, 1.05])
    plt.savefig(os.path.join(SAVE_DIR, f"model_comp_mcc_{dataset_name}.png"), bbox_inches='tight')
    plt.show()

print("========== MODEL KARŞILAŞTIRMALARI (MCC) ==========")
plot_model_comparison_mcc('rayleigh', 'Rayleigh Fading')
plot_model_comparison_mcc('rician_k3', 'Rician Fading (K=3)')
plot_model_comparison_mcc('rician_k10', 'Rician Fading (K=10)')


In [ ]:
# 6b. Kanal Karşılaştırması (MCC)
def plot_channel_comparison_mcc(model_name, model_title):
    rayleigh_mcc  = load_pkl_data(model_name, 'rayleigh',   'mcc_metric_scores.pkl')
    rician3_mcc   = load_pkl_data(model_name, 'rician_k3',  'mcc_metric_scores.pkl')
    rician10_mcc  = load_pkl_data(model_name, 'rician_k10', 'mcc_metric_scores.pkl')
    if rayleigh_mcc is None or rician3_mcc is None or rician10_mcc is None: return

    snrs_sorted = sorted(list(rayleigh_mcc.keys()))
    plt.figure(figsize=(10, 6), dpi=150)
    plt.plot(snrs_sorted, [rayleigh_mcc[s]  for s in snrs_sorted], 'o-', linewidth=2.5, label='Rayleigh',    color='#E63946')
    plt.plot(snrs_sorted, [rician3_mcc[s]   for s in snrs_sorted], 's-', linewidth=2.5, label='Rician K=3',  color='#2A9D8F')
    plt.plot(snrs_sorted, [rician10_mcc[s]  for s in snrs_sorted], '^-', linewidth=2.5, label='Rician K=10', color='#457B9D')
    plt.xlabel('SNR (dB)', fontsize=12)
    plt.ylabel('MCC', fontsize=12)
    plt.title(f'Channel Fading Impact on {model_title} (MCC)', fontsize=14, fontweight='bold')
    plt.legend(fontsize=12, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([-0.1, 1.05])
    plt.savefig(os.path.join(SAVE_DIR, f'channel_comp_mcc_{model_name}.png'), bbox_inches='tight')
    plt.show()

print('========== KANAL KARSILASTIRMALARI (MCC) ==========')
plot_channel_comparison_mcc('mcldnn',   'MCLDNN')
plot_channel_comparison_mcc('petcgdnn', 'PET-CGDNN')


In [ ]:
# 7. İstatistiksel Karşılaştırma Raporu (Yazılı Çıktı)
print("========== YAZILI PERFORMANS ÖZET RAPORU ==========\n")
for dataset in datasets:
    print(f"[{dataset.upper()}] İÇİN GENEL ÖZET:")
    for model in models:
        acc_data = load_pkl_data(model, dataset, 'acc.pkl')
        f1_data = load_pkl_data(model, dataset, 'f1_macro_scores.pkl')
        if acc_data and f1_data:
            avg_acc = np.mean(list(acc_data.values())) * 100
            max_acc = np.max(list(acc_data.values())) * 100
            avg_f1 = np.mean(list(f1_data.values()))
            print(f"  - {model.upper():<8} : Ort. Acc= %{avg_acc:.2f} | Max Acc= %{max_acc:.2f} | Ort. F1= {avg_f1:.4f}")
    print("-" * 60)


## D) Fine-Tuning Training History (Egitim Egrileri)

In [ ]:
# 8. Fine-Tuning Egitim Gecmisi (CSV'den)
import pandas as pd

def plot_finetune_history(model_name, dataset_name):
    csv_path = os.path.join(RESULTS_DIR, f'{model_name}_{dataset_name}', 'finetune_log.csv')
    if not os.path.exists(csv_path):
        print(f'Bulunamadi: {csv_path}')
        return
    df = pd.read_csv(csv_path)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), dpi=150)

    ax1.plot(df['epoch'], df['loss'],     label='Train Loss', linewidth=2, color='#E63946')
    ax1.plot(df['epoch'], df['val_loss'], label='Val Loss',   linewidth=2, color='#457B9D', linestyle='--')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title(f'Loss  |  {model_name.upper()} / {dataset_name}')
    ax1.legend(); ax1.grid(True, linestyle='--', alpha=0.6)

    acc_col     = 'accuracy'     if 'accuracy'     in df.columns else 'acc'
    val_acc_col = 'val_accuracy' if 'val_accuracy' in df.columns else 'val_acc'
    ax2.plot(df['epoch'], df[acc_col],     label='Train Acc', linewidth=2, color='#E63946')
    ax2.plot(df['epoch'], df[val_acc_col], label='Val Acc',   linewidth=2, color='#457B9D', linestyle='--')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
    ax2.set_title(f'Accuracy  |  {model_name.upper()} / {dataset_name}')
    ax2.legend(); ax2.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'history_{model_name}_{dataset_name}.png'), bbox_inches='tight')
    plt.show()

print('========== FINE-TUNING EGITIM GECMISI ==========')
for model in models:
    for dataset in datasets:
        plot_finetune_history(model, dataset)


## E) AWGN Baseline vs Before Fine-Tuning vs After Fine-Tuning

In [ ]:
# 9. AWGN Baseline vs Pre-Fine-Tuned vs Fine-Tuned Karsilastirmasi
AWGN_RESULTS_DIR = '/content/drive/MyDrive/AMR-Project/results'
PRE_FT_RESULTS_DIR = RESULTS_DIR  # fine_tuning_results icinde *_pretrain klasorleri

def load_awgn_data(model_name, metric_file='acc.pkl'):
    for folder in [f'{model_name}_awgn', model_name]:
        for subdir in ['predictions', '']:
            p = os.path.join(AWGN_RESULTS_DIR, folder, subdir, metric_file).replace('//', '/')
            if os.path.exists(p):
                with open(p, 'rb') as f: return pickle.load(f)
    return None

def load_pretrain_data(model_name, dataset_name, metric_file='acc.pkl'):
    """Fine-tuning oncesi: AWGN modeli faded veri uzerinde direkt test sonuclari."""
    folder = f'{model_name}_{dataset_name}_pretrain'
    for subdir in ['predictions', '']:
        p = os.path.join(PRE_FT_RESULTS_DIR, folder, subdir, metric_file).replace('//', '/')
        if os.path.exists(p):
            with open(p, 'rb') as f: return pickle.load(f)
    return None

def plot_three_way(model_name, dataset_name, metric_file, ylabel, title_metric):
    awgn_data = load_awgn_data(model_name, metric_file)
    pre_data  = load_pretrain_data(model_name, dataset_name, metric_file)
    ft_data   = load_pkl_data(model_name, dataset_name, metric_file)

    if ft_data is None:
        print(f'Fine-tuned veri yok: {model_name}/{dataset_name}'); return

    snrs_sorted = sorted(ft_data.keys())
    scale   = 100  if metric_file == 'acc.pkl' else 1
    ylim_hi = 105  if metric_file == 'acc.pkl' else 1.05

    plt.figure(figsize=(10, 6), dpi=150)

    if awgn_data is not None:
        awgn_vals = [awgn_data.get(s, float('nan')) * scale for s in snrs_sorted]
        plt.plot(snrs_sorted, awgn_vals, 'o--', linewidth=2.5,
                 label='AWGN Baseline (AWGN test)', color='#6C757D')

    if pre_data is not None:
        pre_vals = [pre_data.get(s, float('nan')) * scale for s in snrs_sorted]
        plt.plot(snrs_sorted, pre_vals, '^:', linewidth=2.5,
                 label='Before Fine-Tuning (faded test)', color='#F4A261')
    else:
        print(f'  [!] Pre-FT veri yok: {model_name}/{dataset_name} — nb08 tekrar calistirilmali')

    ft_vals = [ft_data[s] * scale for s in snrs_sorted]
    plt.plot(snrs_sorted, ft_vals, 's-', linewidth=2.5,
             label=f'After Fine-Tuning (faded test)', color='#E63946')

    plt.xlabel('SNR (dB)', fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.title(
        f'{model_name.upper()} | {dataset_name} | {title_metric}\n'
        'AWGN Baseline vs Before Fine-Tuning vs After Fine-Tuning',
        fontsize=12, fontweight='bold'
    )
    plt.legend(fontsize=11, loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.ylim([0, ylim_hi])
    fname = f'three_way_{model_name}_{dataset_name}_{metric_file[:-4]}.png'
    plt.savefig(os.path.join(SAVE_DIR, fname), bbox_inches='tight')
    plt.show()

print('========== AWGN BASELINE vs BEFORE FT vs AFTER FT ==========')
metrics_cfg = [
    ('acc.pkl',               'Classification Accuracy (%)', 'Accuracy'),
    ('f1_macro_scores.pkl',   'F1 Score (Macro)',             'F1'),
    ('mcc_metric_scores.pkl', 'MCC',                          'MCC'),
]
for model in models:
    for dataset in datasets:
        for mfile, ylabel, title_m in metrics_cfg:
            plot_three_way(model, dataset, mfile, ylabel, title_m)


## F) Per-Modulation Accuracy Heatmap (SNR x Modulasyon Tipi)

In [ ]:
# 10. Per-Modulation Accuracy Heatmap
import seaborn as sns

MODS = ['8PSK','AM-DSB','AM-SSB','BPSK','CPFSK','GFSK','PAM4','QAM16','QAM64','QPSK','WBFM']
SNR_LABELS = list(range(-20, 20, 2))  # RML2016 SNR aralik: -20..+18 dB

def plot_per_mod_heatmap(model_name, dataset_name, model_title, dataset_title):
    path = os.path.join(RESULTS_DIR, f'{model_name}_{dataset_name}', 'predictions', 'acc_mod_snr.pkl')
    if not os.path.exists(path):
        print(f'Bulunamadi: {path}'); return
    with open(path, 'rb') as f:
        acc_mod_snr = pickle.load(f)

    n_mods, n_snrs = acc_mod_snr.shape
    snr_labels = SNR_LABELS[:n_snrs]
    mod_labels = MODS[:n_mods]

    import pandas as pd
    df_heat = pd.DataFrame(acc_mod_snr * 100, index=mod_labels, columns=snr_labels)

    plt.figure(figsize=(14, 5), dpi=150)
    sns.heatmap(df_heat, annot=True, fmt='.0f', cmap='YlOrRd',
                linewidths=0.4, linecolor='gray',
                vmin=0, vmax=100,
                cbar_kws={'label': 'Accuracy (%)'})
    plt.title(f'Per-Modulation Accuracy (%) | {model_title} / {dataset_title}', fontsize=13, fontweight='bold')
    plt.xlabel('SNR (dB)', fontsize=11)
    plt.ylabel('Modulation Type', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'heatmap_{model_name}_{dataset_name}.png'), bbox_inches='tight')
    plt.show()

print('========== PER-MODULATION ACCURACY HEATMAP ==========')
dataset_titles = {'rayleigh': 'Rayleigh', 'rician_k3': 'Rician K=3', 'rician_k10': 'Rician K=10'}
model_titles   = {'mcldnn': 'MCLDNN', 'petcgdnn': 'PET-CGDNN'}
for model in models:
    for dataset in datasets:
        plot_per_mod_heatmap(model, dataset, model_titles[model], dataset_titles[dataset])
